# Chapter 4 — Linear Models and Their Geometry
**Math for ML Engineers** · companion notebook

This notebook mirrors §4.1–§4.4 of the chapter: the normal equations for ordinary least squares, Ridge regression as MAP estimation under a Gaussian prior, Lasso via the ISTA proximal-gradient algorithm, and gradient-based logistic/softmax classifiers. **Key takeaway: L2 (Ridge) and L1 (Lasso) penalties are not just "regularization knobs" — they correspond to different priors on the weights, and L1's non-differentiability at zero is precisely what induces exact sparsity, the same mechanism that underlies modern weight pruning.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## From OLS to Ridge to Lasso

Ordinary least squares minimizes $\|Xw - y\|_2^2$, giving the closed-form **normal equations**

$$
w^\* = (X^\top X)^{-1} X^\top y .
$$

Adding an $\ell_2$ penalty on the weights gives **Ridge regression**,

$$
w^\*_{\text{ridge}} = \arg\min_w \ \|Xw - y\|_2^2 + \lambda \|w\|_2^2
\;=\; (X^\top X + \lambda I)^{-1} X^\top y ,
$$

which is exactly the MAP estimate of $w$ under a Gaussian likelihood and a Gaussian prior $w \sim \mathcal{N}(0, \tau^2 I)$ (§4.2). Because the $\ell_2$ penalty is smooth, Ridge shrinks coefficients toward zero but essentially never sets them *exactly* to zero.

**Lasso** replaces the penalty with an $\ell_1$ norm,

$$
w^\*_{\text{lasso}} = \arg\min_w \ \tfrac{1}{2}\|Xw - y\|_2^2 + \lambda \|w\|_1 ,
$$

which has no closed form because $\|w\|_1$ is non-differentiable at $w_i = 0$. §4.3 (Lasso/ISTA) derives the fix: **ISTA** (Iterative Shrinkage-Thresholding Algorithm) alternates a gradient step on the smooth part with a proximal step — the **soft-thresholding operator**

$$
S_\tau(x) = \mathrm{sign}(x)\,\max(|x| - \tau,\, 0),
$$

applied elementwise with step size $\eta = 1/L$ where $L = \|X\|_2^2$ (the squared spectral norm) bounds the Lipschitz constant of the least-squares gradient. It is this shrink-and-threshold step that drives small coefficients to *exactly* zero, giving Lasso its sparsity-inducing property.

In [ ]:
# Sparse ground-truth regression problem: d=40 features, only 6 truly active
n, d = 200, 40
k_active = 6

w_true = np.zeros(d)
active_idx = np.random.choice(d, size=k_active, replace=False)
w_true[active_idx] = np.random.choice([-3.0, -2.0, -1.5, 1.5, 2.0, 3.0], size=k_active, replace=False)

X = np.random.randn(n, d)
noise = 0.5 * np.random.randn(n)
y = X @ w_true + noise

print(f"n={n} samples, d={d} features, {k_active} nonzero true coefficients")
print("active indices:", sorted(active_idx.tolist()))

## Implementing normal equations, Ridge, and ISTA

Below are the three estimators exactly as specified in the chapter's companion module:

- `normal_equations` — direct OLS solve via `np.linalg.solve` (never explicitly inverting $X^\top X$).
- `ridge` — same solve with $\lambda I$ added to the Gram matrix.
- `soft_threshold` + `lasso_ista` — the proximal-gradient loop: at each iteration take a gradient step on the least-squares term, then apply $S_{\eta\lambda}(\cdot)$ elementwise. We also track the Lasso objective $\tfrac12\|Xw-y\|_2^2 + \lambda\|w\|_1$ at every iteration to visualize convergence.

In [ ]:
def normal_equations(X, y):
    """w* = (X^T X)^{-1} X^T y — closed-form OLS solution."""
    return np.linalg.solve(X.T @ X, X.T @ y)


def ridge(X, y, lam):
    """Ridge/L2: w* = (X^T X + λI)^{-1} X^T y. Equivalent to MAP under Gaussian prior."""
    n, d = X.shape
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)


def soft_threshold(x, tau):
    """S_tau(x) = sign(x) * max(|x|-tau, 0)."""
    return np.sign(x) * np.maximum(np.abs(x) - tau, 0.0)


def lasso_objective(X, y, w, lam):
    return 0.5 * np.sum((X @ w - y) ** 2) + lam * np.sum(np.abs(w))


def lasso_ista(X, y, lam, n_iters=500):
    """ISTA for Lasso: min_w 0.5||Xw-y||^2 + lam*||w||_1. Step eta=1/L, L=spectral norm^2."""
    n, d = X.shape
    L = np.linalg.norm(X, ord=2) ** 2
    eta = 1.0 / L
    w = np.zeros(d)
    obj_history = np.zeros(n_iters)
    for t in range(n_iters):
        grad = X.T @ (X @ w - y)
        w = soft_threshold(w - eta * grad, eta * lam)
        obj_history[t] = lasso_objective(X, y, w, lam)
    return w, obj_history


lam = 20.0
w_ols = normal_equations(X, y)
w_ridge = ridge(X, y, lam)
w_lasso, obj_history = lasso_ista(X, y, lam, n_iters=1000)

print("nonzeros — true:", np.sum(w_true != 0),
      " OLS:", np.sum(np.abs(w_ols) > 1e-8),
      " Ridge:", np.sum(np.abs(w_ridge) > 1e-8),
      " Lasso:", np.sum(np.abs(w_lasso) > 1e-8))

## Visualizing sparse recovery and convergence

Two plots below make the shrinkage story concrete:

1. A **stem plot** overlaying the true sparse coefficients against Lasso's recovered coefficients — with $\lambda$ large enough relative to the noise level, ISTA should zero out every inactive coefficient and recover the correct *support* (the set of nonzero indices), not just approximately small values.
2. A **convergence plot** of the ISTA objective $\tfrac12\|Xw-y\|_2^2 + \lambda\|w\|_1$ versus iteration, confirming the proximal-gradient step size $\eta = 1/L$ gives monotone descent.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

idx = np.arange(d)
ax = axes[0]
ax.stem(idx - 0.12, w_true, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="true $w$")
ax.stem(idx + 0.12, w_lasso, linefmt="C3-", markerfmt="C3x", basefmt=" ", label="Lasso (ISTA) $\\hat{w}$")
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xlabel("coefficient index")
ax.set_ylabel("value")
ax.set_title("Sparse recovery: true vs. ISTA-recovered coefficients")
ax.legend()

ax = axes[1]
ax.plot(obj_history, color="C3")
ax.set_xlabel("ISTA iteration")
ax.set_ylabel(r"$\frac{1}{2}\|Xw-y\|_2^2 + \lambda\|w\|_1$")
ax.set_title("ISTA objective convergence")
ax.set_yscale("log")

plt.tight_layout()
plt.show()

true_support = set(np.flatnonzero(w_true))
lasso_support = set(np.flatnonzero(np.abs(w_lasso) > 1e-6))
print(f"true support:  {sorted(true_support)}")
print(f"lasso support: {sorted(lasso_support)}")
print(f"exact support match: {true_support == lasso_support}")
print(f"final objective: {obj_history[-1]:.4f}")
print(f"OLS coefficient L2 error:   {np.linalg.norm(w_ols - w_true):.3f}")
print(f"Ridge coefficient L2 error: {np.linalg.norm(w_ridge - w_true):.3f}")
print(f"Lasso coefficient L2 error: {np.linalg.norm(w_lasso - w_true):.3f}")

## Why this matters at production scale: pruning is soft-thresholding

The soft-thresholding operator $S_\tau(x) = \mathrm{sign}(x)\max(|x|-\tau, 0)$ you just used to zero out 34 of 40 coefficients **is the same shrinkage operator that underlies L1-regularized weight pruning of neural networks**, including post-training sparsification of large language models. Magnitude-based pruning — "zero out any weight with $|w_i|$ below a threshold" — is literally the $\lambda \to \infty$, single-step limit of ISTA's proximal step applied directly to the weights instead of to a regression residual. Structured pruning (dropping whole neurons/attention heads/channels) is the group-sparse generalization of the same idea (an $\ell_{2,1}$ penalty instead of plain $\ell_1$), used to compress LLMs after pretraining without full retraining.

There is a second, equally important production lesson hiding in this demo. Ridge's closed-form solution $w^\* = (X^\top X + \lambda I)^{-1} X^\top y$ required solving a $d \times d$ linear system — fine for $d=40$, and even fine for $d$ in the thousands. But a modern LLM has $d$ in the **billions** of parameters: forming, let alone inverting, a $d \times d$ Gram matrix is completely infeasible ($O(d^3)$ time, $O(d^2)$ memory). This is precisely why practitioners never solve Ridge's normal equations at LLM scale — instead they use **first-order stochastic optimizers (Adam) with *decoupled* weight decay (AdamW)**, which applies an L2-like shrinkage $w \leftarrow (1-\eta\lambda) w$ directly in the parameter update rather than folding it into the loss gradient. AdamW's decoupled decay is the $O(d)$-per-step, matrix-inversion-free analogue of exactly the Ridge shrinkage you computed above in closed form.

In [ ]:
# (a) Magnitude pruning of Lasso's *dense* OLS solution reproduces ISTA's sparsity pattern,
#     illustrating "pruning == one-shot soft-thresholding".
def magnitude_prune(w, tau):
    """One-shot hard variant: same support-selection idea as soft-thresholding,
    used directly on trained weights instead of a regression residual."""
    w_pruned = w.copy()
    w_pruned[np.abs(w_pruned) < tau] = 0.0
    return w_pruned

# Sweep a threshold on the dense OLS weights until sparsity matches the Lasso solution
target_nnz = np.sum(np.abs(w_lasso) > 1e-6)
thresholds = np.linspace(0, np.abs(w_ols).max(), 200)
nnz_curve = [np.sum(np.abs(magnitude_prune(w_ols, t)) > 1e-8) for t in thresholds]
best_t = thresholds[np.argmin(np.abs(np.array(nnz_curve) - target_nnz))]
w_pruned = magnitude_prune(w_ols, best_t)

pruned_support = set(np.flatnonzero(np.abs(w_pruned) > 1e-8))
print(f"Lasso support:         {sorted(lasso_support)}")
print(f"magnitude-pruned OLS:  {sorted(pruned_support)}  (threshold={best_t:.3f})")
print(f"support agreement with Lasso: {len(lasso_support & pruned_support)}/{len(lasso_support)}")

# (b) Cost of solving Ridge's (X^T X + lam I) system vs. one AdamW-style decoupled-decay
#     update, as d scales toward LLM territory.
print("\ncost of forming/solving a dxd Gram system (Ridge normal equations) vs.")
print("one O(d) decoupled weight-decay step (AdamW), by parameter count d:")
for d_model in [1e3, 1e6, 1e9, 1.75e11]:  # 175B ~ GPT-3 scale
    gram_flops = d_model ** 3          # O(d^3) solve
    adamw_flops = 4 * d_model          # O(d): elementwise moment updates + decay
    label = f"d={d_model:.0e}"
    print(f"  {label:>10s}:  Gram solve ~ {gram_flops:.2e} FLOPs   "
          f"AdamW step ~ {adamw_flops:.2e} FLOPs   "
          f"ratio ~ {gram_flops/adamw_flops:.2e}x")

## Exercise

1. Derive the soft-thresholding operator $S_\tau(x) = \mathrm{sign}(x)\max(|x|-\tau,0)$ from the proximal operator of $\lambda|w_i|$, i.e. solve $\arg\min_{w_i} \tfrac{1}{2\eta}(w_i - v_i)^2 + \lambda|w_i|$ by splitting into cases $w_i > 0$, $w_i < 0$, $w_i = 0$. Compare your derivation's case-by-case structure to the subgradient conditions for $\ell_1$ optimality derived in §4.3 (the soft-threshold operator, Eq. (4.16)).

2. Rerun the sparse-recovery experiment above while sweeping $\lambda$ over a log-spaced grid (e.g. `np.logspace(-2, 1, 20)`), and plot the number of recovered nonzero coefficients against $\lambda$. At what point does Lasso start *dropping true positives* (zeroing out coefficients that are actually nonzero in `w_true`)? Relate this bias-vs-sparsity tradeoff to the Lasso's shrinkage bias discussed in §4.3, and explain why practitioners often use "debiased Lasso" (OLS refit on the recovered support) to correct it.